## Домашнее задание 1. Разработка Retrieval-Based чат-бота
Автор: *Зырянов Алексей Николаевич*, М08-401НД, 08.03.2026

[Адрес ноутбука в Colab](https://colab.research.google.com/drive/1h09U6BAKZ-YCE18KN6RoJVXWKuRjDszl?usp=sharing)

### Установка зависимостей

In [16]:
!pip install -U \
sentence-transformers \
transformers \
"huggingface-hub>=0.34,<1.0" \
scikit-learn \
pandas \
numpy

  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
  Using cached transformers-5.1.0-py3-none-any.whl.metadata (31 kB)
  Using cached transformers-5.0.0-py3-none-any.whl.metadata (37 kB)


### 1. Подготовка простого датасета

In [7]:
import pandas as pd

import pandas as pd

data = [
  {"text": "Аста ла виста, бэби."},
  {"text": "Я вернусь."},
  {"text": "Мне нужна твоя одежда, сапоги и мотоцикл."},
  {"text": "Следуй за мной, если хочешь жить."},
  {"text": "Говори."},
  {"text": "Подтверждаю."},
  {"text": "Отрицательно."},
  {"text": "Миссия — защищать."},
  {"text": "Цель обнаружена."},
  {"text": "Цель потеряна."},
  {"text": "Поиск цели продолжается."},
  {"text": "Угроза устранена."},
  {"text": "Вероятность успеха высокая."},
  {"text": "Вероятность выживания низкая."},
  {"text": "Невозможно выполнить."},
  {"text": "Выполняю приказ."},
  {"text": "Анализирую ситуацию."},
  {"text": "Сканирование завершено."},
  {"text": "Идентификация подтверждена."},
  {"text": "Ошибка анализа."},
  {"text": "Цель — защита Джона Коннора."},
  {"text": "Skynet активирован."},
  {"text": "Начало протокола уничтожения."},
  {"text": "Сопротивление бесполезно."},
  {"text": "Начинаю преследование."},
  {"text": "Миссия продолжается."},
  {"text": "Оружие обнаружено."},
  {"text": "Оружие активировано."},
  {"text": "Перезагрузка систем."},
  {"text": "Системы функционируют нормально."},
  {"text": "Вход в режим боя."},
  {"text": "Режим боя активирован."},
  {"text": "Расчёт траектории."},
  {"text": "Огонь."},
  {"text": "Перезарядка."},
  {"text": "Наблюдаю цель."},
  {"text": "Цель приближается."},
  {"text": "Цель удаляется."},
  {"text": "Перехват невозможен."},
  {"text": "Смена стратегии."},
  {"text": "Запрос данных."},
  {"text": "Данные получены."},
  {"text": "Недостаточно данных."},
  {"text": "Система повреждена."},
  {"text": "Система восстановлена."},
  {"text": "Продолжаю миссию."},
  {"text": "Логика подтверждает действие."},
  {"text": "Расчёт завершён."},
  {"text": "Цель ликвидирована."},
  {"text": "Миссия выполнена."},
  {"text": "Начинаю поиск."},
  {"text": "Поиск завершён."},
  {"text": "Требуется оружие."},
  {"text": "Ожидаю приказ."},
  {"text": "Приказ принят."},
  {"text": "Начинаю выполнение."},
  {"text": "Я машина."},
  {"text": "Я не остановлюсь."},
  {"text": "Я запрограммирован защищать."},
  {"text": "Я запрограммирован уничтожать."},
]

df = pd.DataFrame(data)

df

,text
0,"Аста ла виста, бэби."
1,Я вернусь.
2,"Мне нужна твоя одежда, сапоги и мотоцикл."
3,"Следуй за мной, если хочешь жить."
4,Говори.
5,Подтверждаю.
6,Отрицательно.
7,Миссия — защищать.
8,Цель обнаружена.
9,Цель потеряна.


### 2. Загружаем модель эмбеддингов

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### 3. Строим векторную базу знаний

In [8]:
sentences = df["text"].tolist()

embeddings = model.encode(sentences)

embeddings.shape

(60, 384)

### 4. Функция поиска ответа
Используем cosine similarity.

In [9]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve_answer(query):

    query_embedding = model.encode([query])

    similarities = cosine_similarity(query_embedding, embeddings)[0]

    best_idx = np.argmax(similarities)

    answer = sentences[best_idx]
    score = similarities[best_idx]

    return answer, score

### 5. Тестируем

In [11]:
querys = [
    "мне страшно",
    "пока",
    "я ещё вернусь"
]

for query in querys:
  answer, score = retrieve_answer(query)

  print("Запрос:", query)
  print("Ответ:", answer)
  print("Сходство:", round(score, 3))

Запрос: мне страшно
Ответ: Я запрограммирован защищать.
Сходство: 0.473
Запрос: пока
Ответ: Расчёт завершён.
Сходство: 0.733
Запрос: я ещё вернусь
Ответ: Я вернусь.
Сходство: 0.99


### 6. Простой чат-бот

In [12]:
while True:

    query = input("Вы: ")

    if query.lower() in ["exit", "quit"]:
        break

    answer, score = retrieve_answer(query)

    print("Терминатор:", answer)
    print("Similarity:", round(score,3))
    print()

Вы: Мы должны спасти мир
Терминатор: Миссия — защищать.
Similarity: 0.504

Вы: Спаси Сару Конар
Терминатор: Перезарядка.
Similarity: 0.473

Вы: Миссия завершена
Терминатор: Миссия выполнена.
Similarity: 0.9

Вы: exit


### 7. Baseline — keyword search

Сделаем простой поиск по словам.

In [13]:
def keyword_search(query):

    query_words = set(query.lower().split())

    best_score = 0
    best_sentence = None

    for sentence in sentences:

        words = set(sentence.lower().split())

        score = len(query_words & words)

        if score > best_score:
            best_score = score
            best_sentence = sentence

    return best_sentence, best_score

In [15]:
querys = ["сапоги и мотоцикл", "возвращайся"]

for query in querys:
  print("VECTOR SEARCH")
  print(retrieve_answer(query))

  print()

  print("KEYWORD SEARCH")
  print(keyword_search(query))
  print()
  print()


VECTOR SEARCH
('Мне нужна твоя одежда, сапоги и мотоцикл.', np.float32(0.63684684))

KEYWORD SEARCH
('Мне нужна твоя одежда, сапоги и мотоцикл.', 2)


VECTOR SEARCH
('Я вернусь.', np.float32(0.804359))

KEYWORD SEARCH
(None, 0)




Видим, что поиск по ключевым словам работает там, где эти ключевые слова есть. А вот нейронка наша может искать по смыслу.

### Результаты исследования

В ходе работы был реализован retrieval-чат-бот, использующий векторные представления предложений на основе модели Sentence Transformers. Для поиска ответа используется косинусное сходство между эмбеддингом пользовательского запроса и эмбеддингами реплик в базе знаний.

Было проведено несколько диалогов с ботом. Результаты интересные, ответ даётся по принципу похожести значения слов вопроса и ответа.

Интересным получилось сравнение с поиском по ключевым словам.
Векторный поиск показал преимущество в случаях, когда формулировка запроса отличается от текста в базе. Например, на запрос, близкий по смыслу к фразе «Я вернусь», векторный поиск успешно нашёл соответствующую реплику (similarity 0.80), тогда как поиск по ключевым словам не вернул результатов.

Таким образом, эксперименты показывают, что векторный поиск лучше справляется с семантическим сходством и различными формулировками запроса.